In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline
import pandas as pd
import numpy as np
import os
import sys
from tqdm import tqdm

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- 配置 ---
squence = 'MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGTYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFKIRHNIVEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSVLSKDPNEKRDHMVLLEFVTAAGITHGMDELYK'

MODEL_DIR = './esm2_150M_origin/'

# --- 自动构建文件路径 (无需修改) ---
DEVICE = 0 if torch.cuda.is_available() else -1
TOP_K_PREDICTIONS = 1
OUTPUT_FILE = "./prediction_results.csv"
COLUMN_SEQUENCES = 'sequence'
MAX_MUTATION_SITE_NUM = 1

# --- 预测参数 ---
confidence_threshold = 0.8
MAX_PREDICTIONS_LIMIT = 500
SCAN_BATCH_SIZE = 4


def process_pipeline_results(pipeline_output, original_sequence, original_aas):
    """
    解析 pipeline 的原始输出，生成预测序列和差异分析文本。
    """
    analysis = ''
    predicted_seq = list(original_aas)
    different_cnt = 0
    candidate_mutation_sites = []
    
    if len(pipeline_output) != len(original_sequence):
        error_msg = f"错误: Pipeline 输出长度 ({len(pipeline_output)}) 与原始序列长度 ({len(original_sequence)}) 不匹配。"
        print(error_msg)
        return "PROCESSING_ERROR", error_msg

    for i, single_pos_predictions in enumerate(pipeline_output):
        assert single_pos_predictions, '预测结果出错, 输出为空'
        top_pred = single_pos_predictions[0]
        
        if top_pred['score'] < confidence_threshold:
            continue
        else:
            if top_pred['token_str'] != original_aas[i]:
                analysis += f"\tposition {i}: predict {top_pred['token_str']} while original {original_aas[i]} (score: {top_pred['score']:.4f}), as a candidate\n"
                different_cnt += 1
                candidate_mutation_sites.append([i, top_pred['token_str'], top_pred['score']])

    candidate_mutation_sites.sort(key=lambda x: -x[2])
    print(candidate_mutation_sites)
    for i in range(min(len(candidate_mutation_sites), MAX_MUTATION_SITE_NUM)):
        predicted_seq[candidate_mutation_sites[i][0]] = candidate_mutation_sites[i][1]
        print('氨基酸替换')
    analysis_header = f'\ttotal different aa count: {different_cnt}\n'
    predicted_seq = ''.join(predicted_seq)
    return predicted_seq, analysis_header + analysis


if __name__ == '__main__':
    model = AutoModelForMaskedLM.from_pretrained("/esm2_150M_origin/")
    try:
        print(f"Loading Model from: {MODEL_DIR}")
        pipeline = pipeline("fill-mask", model=MODEL_DIR, device=DEVICE)
    except Exception as e:
        print(f"FATAL: Could not load model. Error: {e}")
        print(f"Please ensure the path '{MODEL_DIR}' exists and is a valid model directory.")
        sys.exit(1)
    print("--- Model loaded successfully. ---\n")
    tokenizer = pipeline.tokenizer

    # --- 2. 读取和筛选数据 ---
    if not os.path.exists(SEQ_FILE):
        print(f"FATAL: Input file not found at {SEQ_FILE}")
        sys.exit(1)

    # --- 3. 主预测循环 ---
    result_datas = []

    sequence = seqs_df.iloc[i][COLUMN_SEQUENCES]
        
    if len(sequence) > (tokenizer.model_max_length - 2):
        sequence = sequence[:(tokenizer.model_max_length - 2)]
        print(f"  > row {i}'s length is out of limit, cutting only 1024 reisdue")
        
    masked_sequences = []
    original_aas = list(sequence)
    for j in range(len(sequence)):
        masked_list = list(sequence)
        masked_list[j] = tokenizer.mask_token
        masked_sequences.append("".join(masked_list))
            
    # --- 3b. 高效运行 Pipelines ---
    print(f"  > Running finetuned pipeline for {len(masked_sequences)} masks...")
    # 修正2: 直接将列表 `masked_sequences` 传递给 pipeline
    results = pipeline(masked_sequences, batch_size=SCAN_BATCH_SIZE, top_k=TOP_K_PREDICTIONS)

    # --- 3c. 处理结果 ---
    # print(original_aas[:5])
    finetuned_seq, finetuned_analysis = process_pipeline_results(finetuned_results, sequence, original_aas)
    # print(original_aas[:5])
    modified_seq, analysis = process_pipeline_results(base_results, sequence, original_aas)

print(modified_seq)